## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | Adapt the selected SE-ResNeXt checkpoint using paired published and YOLO-ROI views. |
| Model | SE-ResNeXt-50 32x4d |
| Input | 224x224 knee ROI |
| Task | Paired-view YOLO ROI adaptation training |
| Loss | See training cells |
| Configuration | See configuration cells |
| Result | See the executed cells below for metrics, plots, and checkpoint details. |
| Status | training notebook |


# 03 - SE-ResNeXt-50 Paired-View YOLO Adaptation Training

Run notebook 01 first. This is the SE-ResNeXt counterpart of the selected
DenseNet paired-view adaptation, with every data and optimization choice held
constant:

- published 224x224 crop: used 50% of the time
- expanded 1.15x YOLO square ROI: used 50% of the time
- five fine-tuning epochs from the selected CE SE-ResNeXt checkpoint
- CE is the only training loss; no MSE or ordinal loss is introduced
- CLAHE, then square padding, then resize to 224x224

The only intentional difference is the classifier architecture. SE-ResNeXt
retains its final 1x1-convolution native-CAM head so its checkpoint remains
compatible with the application. Validation reports both source views and
selects the checkpoint by their mean selection score. Test data is not used
for tuning.


In [8]:
!pip -q install "timm>=1.0" "h5py>=3.9"


In [9]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Fixed paired-view configuration

`BASE_CHECKPOINT` is the selected CE SE-ResNeXt-50 checkpoint from Notebook 02.
It must have the `final_native_cam_ce` architecture. Do not use a CORN,
ordinal, or classifier-head checkpoint because its state dictionary is not
compatible with the native-CAM model below.


In [10]:
SEED = 42
INPUT_SIZE = 224
BATCH_SIZE = 48
NUM_WORKERS = 2
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-3
ALTERNATE_VIEW_PROBABILITY = 0.50

PUBLISHED_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224")
ROI_ROOT = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2")
BASE_CHECKPOINT = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_checkpoints/2026-08-04_00-59-41_684745_UTC_original_224_ce_3stage/best_model.pth")
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_paired_view_adaptation") / f"{RUN_TIMESTAMP}_paired_view_yolo_roi"

for required in (PUBLISHED_ROOT, ROI_ROOT, BASE_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(required)
RUN_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


## Build paired published/YOLO records

Every published image must have the same patient-side file in the generated ROI folder. The test split is deliberately excluded from adaptation and checkpoint selection.


In [11]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for published_path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            roi_path = ROI_ROOT / split / str(grade) / published_path.name
            if not roi_path.is_file():
                raise FileNotFoundError(f"Missing paired ROI: {roi_path}")
            rows.append({
                "split": split,
                "grade": grade,
                "published_path": str(published_path),
                "roi_path": str(roi_path),
            })
frame = pd.DataFrame(rows)
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))


grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27


## Preprocessing, paired dataset, and model

The alternate ROI is selected independently for each training sample. This is the paired-view adaptation mechanism. It is not MSE feature matching.


In [12]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(image, top, side - height - top, left, side - width - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))


train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50), transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class PairedDataset(Dataset):
    def __init__(self, data, transform, alternate_probability):
        self.data = data.reset_index(drop=True)
        self.transform = transform
        self.alternate_probability = alternate_probability
        self.labels = self.data.grade.astype(int).tolist()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        use_roi = self.alternate_probability > 0 and random.random() < self.alternate_probability
        image = cv2.imread(row.roi_path if use_roi else row.published_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read paired image at index {index}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)


class SEResNeXt50NativeCAM(nn.Module):
    # Same native-CAM head used by the application.

    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "seresnext50_32x4d",
            pretrained=False,
            features_only=True,
            out_indices=(4,),
        )
        self.class_conv = nn.Conv2d(self.backbone.feature_info.channels()[0], 5, kernel_size=1)

    def class_maps(self, images):
        return self.class_conv(self.backbone(images)[0])

    def forward(self, images):
        return self.class_maps(images).mean(dim=(2, 3))


## Fine-tune for five epochs with cross-entropy

This intentionally mirrors the DenseNet paired-view source experiment. Its
robustness comes from alternating published and exact-production ROI views,
then validating on both domains. The SE-ResNeXt native-CAM head is preserved;
there is no MSE, CAM loss, or other architecture-specific training objective.


In [13]:
checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
if checkpoint.get("loss_type") not in (None, "ce"):
    raise RuntimeError(f"Expected CE checkpoint, got {checkpoint.get('loss_type')}")
if checkpoint.get("architecture") not in (None, "final_native_cam_ce", "natural_final_native_cam_ce"):
    raise RuntimeError(f"Expected SE-ResNeXt native-CAM checkpoint, got {checkpoint.get('architecture')}")

model = SEResNeXt50NativeCAM().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
train_frame = frame[frame.split == "train"].copy()
val_frame = frame[frame.split == "val"].copy()
counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)
train_loader = DataLoader(PairedDataset(train_frame, train_transform, ALTERNATE_VIEW_PROBABILITY), batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def evaluate(data, use_roi):
    loader = DataLoader(PairedDataset(data, val_transform, 1.0 if use_roi else 0.0), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    labels, probabilities = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in loader:
            probs = F.softmax(model(images.to(DEVICE, non_blocking=True)).float(), dim=1).cpu().numpy()
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs)
    labels, probabilities = np.asarray(labels), np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    return {"qwk": float(qwk), "macro_f1": float(f1), "macro_ap": float(ap), "selection": float(0.55 * qwk + 0.30 * f1 + 0.15 * ap)}


best_score = -float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = F.cross_entropy(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()
    published = evaluate(val_frame, use_roi=False)
    roi = evaluate(val_frame, use_roi=True)
    robust = 0.5 * (published["selection"] + roi["selection"])
    row = {"epoch": epoch, "train_loss": loss_sum / samples, "robust_selection": robust, **{f"published_{k}": v for k, v in published.items()}, **{f"roi_{k}": v for k, v in roi.items()}}
    history.append(row)
    print(json.dumps(row, indent=2))
    if robust > best_score:
        best_score = robust
        torch.save({"model_state_dict": model.state_dict(), "architecture": "final_native_cam_ce",
            "model_name": "seresnext50_32x4d", "loss_type": "ce", "epoch": epoch, "paired_view_probability": ALTERNATE_VIEW_PROBABILITY, "roi_expansion": 1.15, "robust_selection": robust}, RUN_DIR / "best_model.pth")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
(RUN_DIR / "run_config.json").write_text(json.dumps({"loss": "cross_entropy", "mse_used": False, "alternate_view_probability": ALTERNATE_VIEW_PROBABILITY, "roi_expansion": 1.15, "epochs": EPOCHS, "base_checkpoint": str(BASE_CHECKPOINT), "model_name": "seresnext50_32x4d", "architecture": "final_native_cam_ce", "cam_evaluation": "post_hoc_gradcam"}, indent=2))
print("Best checkpoint:", RUN_DIR / "best_model.pth")


epoch 1/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 0.8089204525279108,
  "robust_selection": 0.666432953119518,
  "published_qwk": 0.7669579692051602,
  "published_macro_f1": 0.6196868648501538,
  "published_macro_ap": 0.6758059163222004,
  "published_selection": 0.7091038299662145,
  "roi_qwk": 0.676503570796199,
  "roi_macro_f1": 0.5531528475539375,
  "roi_macro_ap": 0.5715950537915392,
  "roi_selection": 0.6237620762728215
}


epoch 2/5:   0%|          | 0/121 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ee0abc51f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ee0abc51f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

{
  "epoch": 2,
  "train_loss": 0.755227071721482,
  "robust_selection": 0.6750896811961111,
  "published_qwk": 0.7665129648481945,
  "published_macro_f1": 0.6231250877288265,
  "published_macro_ap": 0.6670981842367093,
  "published_selection": 0.7085843846206614,
  "roi_qwk": 0.6991015326795467,
  "roi_macro_f1": 0.5606220827773409,
  "roi_macro_ap": 0.5926833997640528,
  "roi_selection": 0.6415949777715608
}


...

epoch 4/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "epoch": 4,
  "train_loss": 0.7203844056570147,
  "robust_selection": 0.6719421673507333,
  "published_qwk": 0.7716615698267074,
  "published_macro_f1": 0.6197724042775663,
  "published_macro_ap": 0.6698684404321007,
  "published_selection": 0.710825850752774,
  "roi_qwk": 0.6868808165716,
  "roi_macro_f1": 0.5528121888448174,
  "roi_macro_ap": 0.5962025212057827,
  "roi_selection": 0.6330584839486927
}


epoch 5/5:   0%|          | 0/121 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ee0abc51f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ee0abc51f80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

{
  "epoch": 5,
  "train_loss": 0.7253245171604374,
  "robust_selection": 0.6718147176845429,
  "published_qwk": 0.7635276532137518,
  "published_macro_f1": 0.6241303875998835,
  "published_macro_ap": 0.6680750781974193,
  "published_selection": 0.7073905872771415,
  "roi_qwk": 0.6913263820119998,
  "roi_macro_f1": 0.556846730585603,
  "roi_macro_ap": 0.5930354587310901,
  "roi_selection": 0.6362388480919443
}
Best checkpoint: /content/drive/MyDrive/Models/seresnext50_32x4d_paired_view_adaptation/2026-08-04_05-11-53_194345_UTC_paired_view_yolo_roi/best_model.pth
